# Designing the Agent-Computer Interface

Every agent pattern — routing, ReAct, plan-and-execute — ultimately acts through an **Agent-Computer Interface (ACI)**: the **tool schemas** the model sees and the **observation strings** your code returns after execution.

The model does not call Python directly. It reads JSON tool definitions, emits structured tool calls, and parses text observations. **ACI is UX for models** — and weak ACI makes even strong patterns loop, hallucinate, and burn tokens.

```
  Pattern (routing, ReAct, …)
         │
         ▼
  ┌──────────────────┐
  │  ACI — schemas   │  names, args, descriptions
  └────────┬─────────┘
           ▼
  ┌──────────────────┐
  │  Tool executor   │  your code runs safely
  └────────┬─────────┘
           ▼
  ┌──────────────────┐
  │  Observations    │  STATUS, SUMMARY, HINT
  └──────────────────┘
```

This notebook designs ACI for **ShopCo Support Hub** and **ReleaseFlow**: schemas, observation envelopes, risk tiers, pattern-specific tool surfaces, and a measurable checklist.

In [ ]:
"""
Agent-Computer Interface lab — ShopCo + ReleaseFlow.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable, Literal

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
MAX_AGENT_STEPS = 5


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_DOCS = [
    {"id": "pol-returns-01", "title": "Returns", "text": "Returns within 30 days with receipt."},
    {"id": "pol-shipping-02", "title": "Shipping", "text": "Free shipping on orders over $50."},
]

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped", "total_usd": 89.0},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing", "total_usd": 1299.0},
}

RELEASE_CHECKS = {
    "unit_tests": {"status": "pass", "detail": "412 tests passed"},
    "lint": {"status": "pass", "detail": "no issues"},
    "security_scan": {"status": "warn", "detail": "1 medium CVE"},
}

print("ACI lab ready.")

---

## 1. ACI vs Human-Computer Interface (HCI)

| HCI | ACI |
|-----|-----|
| Buttons, forms | Tool schemas |
| Visual feedback | Observation strings |
| Error toasts | Parseable `[STATUS: error]` + `[HINT]` |
| User mental model | Model context window |

Design ACI assuming the "user" has a **limited attention span** (context) and **no access to your source code**.

---

## 2. Observation Envelope — Consistent Tool Output

In [ ]:
ObservationStatus = Literal["ok", "error", "empty"]


def format_observation(
    tool_name: str,
    status: ObservationStatus,
    body: str,
    hint: str = "",
) -> str:
    summary = (body.split("\n")[0] if body else "(no content)")[:80]
    lines = [
        f"[STATUS: {status}]",
        f"[TOOL: {tool_name}]",
        f"[SUMMARY: {summary}]",
        "---",
        body,
    ]
    if hint:
        lines.append(f"[HINT: {hint}]")
    return "\n".join(lines)


print(format_observation("search_policy", "ok", "[pol-returns-01] Returns within 30 days.", hint="Cite pol-returns-01 in reply."))

---

## 3. Tool Schema Model

Schemas are the **input side** of ACI. Good schemas use verb names, typed parameters, and descriptions that say **when** to use the tool.

In [ ]:
@dataclass
class ToolSchema:
    name: str
    description: str
    parameters: dict[str, Any]
    risk_tier: str = "read"  # read | write | admin
    patterns: list[str] = field(default_factory=list)

    def to_openai_tool(self) -> dict[str, Any]:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.parameters,
            },
        }


SHOPCO_TOOLS: list[ToolSchema] = [
    ToolSchema(
        "get_order",
        "Look up order status by order_id. Use when customer mentions ORD-####.",
        {"type": "object", "properties": {"order_id": {"type": "string", "description": "Format ORD-####"}}, "required": ["order_id"]},
        patterns=["routing", "react", "plan_and_execute"],
    ),
    ToolSchema(
        "search_policy",
        "Search ShopCo policy docs. Use for returns, shipping, billing questions.",
        {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
        patterns=["routing", "react", "multi_agent"],
    ),
]

RELEASE_TOOLS: list[ToolSchema] = [
    ToolSchema(
        "run_ci_check",
        "Run a named CI check: unit_tests, lint, or security_scan.",
        {"type": "object", "properties": {"check_name": {"type": "string"}}, "required": ["check_name"]},
        patterns=["parallelization", "plan_and_execute"],
    ),
    ToolSchema(
        "deploy_release",
        "Deploy version to target environment. Requires human approval for production.",
        {"type": "object", "properties": {"version": {"type": "string"}, "target": {"type": "string"}}, "required": ["version", "target"]},
        risk_tier="admin",
        patterns=["plan_and_execute"],
    ),
]

print("ShopCo tools:", [t.name for t in SHOPCO_TOOLS])

---

## 4. Pattern → ACI Surface Matrix

Different patterns need different **action spaces**:

| Pattern | ACI design focus |
|---------|------------------|
| **Routing** | Small tool set per route; hide irrelevant tools |
| **ReAct** | Broader tool palette; strong observation hints |
| **Plan-and-execute** | Named actions map 1:1 to plan steps |
| **Parallelization** | Idempotent read tools; merge-friendly output |
| **Multi-agent** | Per-agent tool ACLs; shared read-only resources |

In [ ]:
def tools_for_pattern(pattern: str, domain: str = "shopco") -> list[ToolSchema]:
    pool = SHOPCO_TOOLS if domain == "shopco" else RELEASE_TOOLS
    if pattern == "routing":
        return [t for t in pool if t.risk_tier == "read"]
    if pattern == "plan_and_execute":
        return pool
    return pool


print("Routing sees:", [t.name for t in tools_for_pattern("routing")])
print("Plan-and-execute sees:", [t.name for t in tools_for_pattern("plan_and_execute", "release")])

---

## 5. ShopCo Tool Implementations (ACI-Wrapped)

In [ ]:
def get_order(order_id: str) -> str:
    oid = order_id.upper().strip()
    if not re.match(r"^ORD-\d{4}$", oid):
        return format_observation(
            "get_order", "error",
            f"Invalid order_id '{order_id}'. Expected ORD-####.",
            hint="Ask the customer to confirm their order ID.",
        )
    rec = ORDERS_DB.get(oid)
    if not rec:
        known = ", ".join(sorted(ORDERS_DB))
        return format_observation(
            "get_order", "error",
            f"Order {oid} not found. Known: {known}.",
            hint="Verify ID or search_policy for general help.",
        )
    return format_observation(
        "get_order", "ok", pretty(rec),
        hint=f"Status is {rec['status']}.",
    )


def search_policy(query: str) -> str:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    hits = []
    for doc in POLICY_DOCS:
        hay = f"{doc['title']} {doc['text']}".lower()
        if any(t in hay for t in terms) if terms else True:
            hits.append(doc)
    if not hits:
        return format_observation(
            "search_policy", "empty",
            "No policy chunks matched.",
            hint="Try query='returns' or 'shipping'.",
        )
    body = "\n".join(f"[{d['id']}] {d['text']}" for d in hits[:3])
    return format_observation(
        "search_policy", "ok", body,
        hint="Cite policy ids like [pol-returns-01] in the customer reply.",
    )


print(get_order("ORD-1001"))
print("\n", search_policy("returns"))

---

## 6. ReleaseFlow Tool Implementations

In [ ]:
def run_ci_check(check_name: str) -> str:
    rec = RELEASE_CHECKS.get(check_name)
    if not rec:
        allowed = ", ".join(RELEASE_CHECKS)
        return format_observation(
            "run_ci_check", "error",
            f"Unknown check '{check_name}'. Allowed: {allowed}.",
        )
    status: ObservationStatus = "ok" if rec["status"] != "fail" else "error"
    return format_observation(
        "run_ci_check", status,
        f"check={check_name} status={rec['status']} detail={rec['detail']}",
        hint="Proceed only if all checks pass or waivers approved.",
    )


def deploy_release(version: str, target: str, human_approved: bool = False) -> str:
    if target == "production" and not human_approved:
        return format_observation(
            "deploy_release", "error",
            "Production deploy blocked — human approval required.",
            hint="Call request_human_approval then retry deploy_release.",
        )
    return format_observation(
        "deploy_release", "ok",
        f"Deployed {version} to {target}.",
    )


print(run_ci_check("security_scan"))
print(deploy_release("2.4.0", "production"))

---

## 7. ACI Tool Registry and Executor

In [ ]:
@dataclass
class ToolCall:
    call_id: str
    name: str
    arguments: dict[str, Any]


@dataclass
class ToolResult:
    call_id: str
    tool_name: str
    observation: str
    latency_ms: int = 0


TOOL_IMPLS: dict[str, Callable[..., str]] = {
    "get_order": lambda **kw: get_order(kw["order_id"]),
    "search_policy": lambda **kw: search_policy(kw["query"]),
    "run_ci_check": lambda **kw: run_ci_check(kw["check_name"]),
    "deploy_release": lambda **kw: deploy_release(kw["version"], kw["target"], kw.get("human_approved", False)),
}


class ACIToolExecutor:
    def __init__(self, impls: dict[str, Callable[..., str]], schemas: list[ToolSchema]) -> None:
        self.impls = impls
        self.schemas = {s.name: s for s in schemas}
        self.audit: list[dict[str, Any]] = []

    def execute(self, call: ToolCall, context: dict[str, Any] | None = None) -> ToolResult:
        ctx = context or {}
        schema = self.schemas.get(call.name)
        if not schema:
            obs = format_observation(call.name, "error", f"Unknown tool '{call.name}'.")
            return ToolResult(call.call_id, call.name, obs)
        if schema.risk_tier == "admin" and not ctx.get("human_approved"):
            obs = format_observation(call.name, "error", "Admin tool blocked without approval.", hint="Request human approval.")
            return ToolResult(call.call_id, call.name, obs)
        fn = self.impls.get(call.name)
        if not fn:
            obs = format_observation(call.name, "error", "Tool not implemented.")
            return ToolResult(call.call_id, call.name, obs)
        try:
            args = {**call.arguments, **{k: v for k, v in ctx.items() if k in ("human_approved",)}}
            obs = fn(**args)
        except (KeyError, TypeError) as exc:
            obs = format_observation(call.name, "error", f"Invalid arguments: {exc}", hint=f"See schema for {call.name}.")
        self.audit.append({"tool": call.name, "args": call.arguments, "ts": utc_now()})
        return ToolResult(call.call_id, call.name, obs)


shopco_executor = ACIToolExecutor(TOOL_IMPLS, SHOPCO_TOOLS)
r = shopco_executor.execute(ToolCall("tc1", "get_order", {"order_id": "ORD-1002"}))
print(r.observation)

---

## 8. Schema Quality Checklist (Automated)

In [ ]:
ACI_CHECKLIST = [
    "Verb-based name (get_order not order)",
    "Description says WHEN to use tool",
    "Required fields listed in parameters",
    "Observations use STATUS/SUMMARY/HINT envelope",
    "Errors are actionable strings not stack traces",
    "Risk tier set for write/admin tools",
    "Responses bounded (no megabyte dumps)",
]


def audit_schema(schema: ToolSchema) -> list[str]:
    issues: list[str] = []
    if " " in schema.name:
        issues.append("name contains spaces")
    if len(schema.description) < 20:
        issues.append("description too short")
    if "required" not in schema.parameters:
        issues.append("missing required array in parameters")
    if schema.risk_tier == "admin" and "approval" not in schema.description.lower():
        issues.append("admin tool should mention approval in description")
    return issues


for t in SHOPCO_TOOLS + RELEASE_TOOLS:
    issues = audit_schema(t)
    print(f"{t.name:18} issues={issues or 'none'}")

---

## 9. Bad vs Good ACI — Side by Side

In [ ]:
BAD_SCHEMA = {
    "name": "data",
    "description": "Gets data",
    "parameters": {"type": "object", "properties": {}},
}

GOOD_SCHEMA = SHOPCO_TOOLS[0].to_openai_tool()

BAD_OBSERVATION = "None"  # Python None serialized poorly
GOOD_OBSERVATION = get_order("ORD-9999")

print("Bad schema:", BAD_SCHEMA["name"], "— vague, no required args")
print("Good schema:", GOOD_SCHEMA["function"]["name"], "—", GOOD_SCHEMA["function"]["description"][:50])
print("\nBad observation:", BAD_OBSERVATION)
print("\nGood observation:\n", GOOD_OBSERVATION)

---

## 10. Human Gate as ACI Primitive

`request_human_approval` is a first-class tool — models learn to pause before admin actions.

In [ ]:
def request_human_approval(action: str, reason: str) -> str:
    return format_observation(
        "request_human_approval", "ok",
        f"Approval requested for: {action}. Reason: {reason}",
        hint="Wait for operator APPROVED before admin tools.",
    )


HITL_TOOLS = RELEASE_TOOLS + [
    ToolSchema(
        "request_human_approval",
        "Pause and ask a human operator to approve a risky action before proceeding.",
        {"type": "object", "properties": {"action": {"type": "string"}, "reason": {"type": "string"}}, "required": ["action", "reason"]},
        risk_tier="read",
    ),
]

release_executor = ACIToolExecutor(
    {**TOOL_IMPLS, "request_human_approval": lambda **kw: request_human_approval(kw["action"], kw["reason"])},
    HITL_TOOLS,
)

print(request_human_approval("deploy 2.4.0 to production", "security_scan warning"))

---

## 11. Offline ACI Agent Loop (ReAct Shape)

In [ ]:
def plan_tool_calls(message: str) -> list[ToolCall]:
    calls: list[ToolCall] = []
    if "return" in message.lower() or "policy" in message.lower():
        calls.append(ToolCall("c1", "search_policy", {"query": "returns"}))
    m = re.search(r"ORD-[0-9]{4}", message.upper())
    if m:
        calls.append(ToolCall("c2", "get_order", {"order_id": m.group(0)}))
    return calls


@dataclass
class ACIRunResult:
    message: str
    tool_results: list[ToolResult]
    answer: str


def run_aci_agent(message: str) -> ACIRunResult:
    executor = shopco_executor
    results: list[ToolResult] = []
    for call in plan_tool_calls(message)[:MAX_AGENT_STEPS]:
        results.append(executor.execute(call))
    snippets = [r.observation.split("---")[-1].strip() for r in results]
    answer = " ".join(snippets) if snippets else "No tools invoked."
    return ACIRunResult(message, results, answer)


aci_run = run_aci_agent("Can I return ORD-1001?")
print("Tools called:", [r.tool_name for r in aci_run.tool_results])
print("Answer:", aci_run.answer[:120], "...")

---

## 12. Truncation — Megabyte Backends, Kilobyte Observations

In [ ]:
MAX_OBS_CHARS = 800


def truncate_observation(obs: str, limit: int = MAX_OBS_CHARS) -> str:
    if len(obs) <= limit:
        return obs
    return obs[:limit] + f"\n[TRUNCATED at {limit} chars — refine query]"


huge = format_observation("search_policy", "ok", "x" * 2000)
print("Before:", len(huge), "chars")
print("After:", len(truncate_observation(huge)), "chars")

---

## 13. Measuring ACI Quality

In [ ]:
@dataclass
class ACIMetrics:
    tool_success_rate: float
    avg_steps_to_answer: float
    parseable_error_rate: float
    citation_in_final: float


TEST_QUERIES = [
    "Return policy for ORD-1001",
    "Status ORD-1002",
    "ORD-9999 status",
]


def evaluate_aci(queries: list[str]) -> ACIMetrics:
    successes = 0
    steps = 0
    citations = 0
    error_count = 0
    parseable = 0
    for q in queries:
        run = run_aci_agent(q)
        steps += len(run.tool_results)
        for tr in run.tool_results:
            if "[STATUS: ok]" in tr.observation:
                successes += 1
            if "[STATUS: error]" in tr.observation:
                error_count += 1
                if "[HINT:" in tr.observation:
                    parseable += 1
        if "[pol-" in run.answer:
            citations += 1
    n = max(len(queries), 1)
    total_calls = max(sum(len(run_aci_agent(q).tool_results) for q in queries), 1)
    return ACIMetrics(
        tool_success_rate=round(successes / total_calls, 2),
        avg_steps_to_answer=round(steps / n, 2),
        parseable_error_rate=round(parseable / max(error_count, 1), 2),
        citation_in_final=round(citations / n, 2),
    )


print(pretty(evaluate_aci(TEST_QUERIES)))

---

## 14. Standardized ACI — Protocol Shape (Conceptual)

Protocols like **MCP** standardize discovery (`tools/list`), invocation (`tools/call`), and resources — but **you still design semantics**: clear names, bounded observations, and risk tiers. ACI quality rules apply regardless of wire format.

---

## 15. Optional Live LLM Tool Selection

Set `USE_LIVE_LLM = True` to let `gpt-4o-mini` pick tools from schemas.

In [ ]:
def select_tools_live(message: str, tools: list[ToolSchema]) -> list[str]:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    names = [t.name for t in tools]
    resp = llm.invoke([
        SystemMessage(content=f"Pick tool names from {names} for this message. Reply JSON list of strings."),
        HumanMessage(content=message),
    ])
    try:
        return json.loads(str(resp.content))
    except json.JSONDecodeError:
        return [t.name for t in plan_tool_calls(message)]


msg = "Can I return order ORD-1001?"
if USE_LIVE_LLM:
    print(select_tools_live(msg, SHOPCO_TOOLS))
else:
    print([c.name for c in plan_tool_calls(msg)])

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Vague tool names (`data`) | Wrong tool picked | Verb + entity (`get_order`) |
| Raw exceptions as observations | Model loops | `format_observation` errors |
| Huge JSON dumps | Context rot | Truncate + summarize |
| Same tools for every pattern | Confused model | Pattern-scoped tool surface |
| Admin tools without gate | Production incidents | `risk_tier` + HITL tool |

---

## 17. Quiz

1. What are the input and output sides of ACI?
2. What four fields appear in the observation envelope?
3. Why expose fewer tools under routing than ReAct?
4. What makes an error message "model-parseable"?
5. How does `risk_tier` affect the executor?

---

## 18. Summary

**Designing the agent-computer interface** means crafting **tool schemas** the model can select reliably and **observations** it can act on — for every pattern you deploy.

ShopCo tools (`get_order`, `search_policy`) and ReleaseFlow tools (`run_ci_check`, `deploy_release`) demonstrate envelopes, parseable errors, risk tiers, and human gates. The `ACIToolExecutor` is the single boundary where all tools return consistent output.

Invest in ACI as seriously as API design — it determines whether agents finish in one turn or five.